<a href="https://colab.research.google.com/github/Rohit-Saini-Sfdc/learn-python/blob/main/01_asyncio_mastery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐍 Python Asyncio & Concurrency Mastery
### *From Synchronous Execution to High-Performance Asynchronous Programming*



## 📌 Module Overview
In this interactive notebook, we explore Python's **`asyncio`** library—a core module for writing single-threaded concurrent code using coroutines, multiplexing I/O access, and building high-performance applications.

### 🎯 Learning Objectives
1. Understand **Synchronous Execution**: How Python runs code line-by-line and why blocking I/O causes CPU idle time.
2. Demystify **Asynchronous Programming (`asyncio`)**: Event loops, Coroutines, Tasks, and Futures.
3. Quantify **Performance & Scalability Benefits**: Compare synchronous execution vs. asynchronous execution.
4. Master **7 Different Usage Patterns** of `asyncio` in modern Python.
5. Learn **Best Practices & Common Pitfalls** to avoid freezing the Event Loop.

---


## 1. ⚙️ Synchronous Execution in Python

### How Synchronous Execution Works
By default, Python code runs **synchronously** in a single thread. The execution flow follows a sequential line-by-line path.

When a program encounters an **I/O-bound operation** (e.g., fetching a webpage, reading a file, querying a database, or sleeping with `time.sleep()`), execution **blocks**. The thread halts and waits for the operation to complete before proceeding to the next line.

```
Synchronous Execution Flow:

[ Task 1 Start ] ---> [ Blocking I/O Wait ⏳ ] ---> [ Task 1 Finish ]
                                                           │
                                                           ▼
[ Task 2 Start ] ---> [ Blocking I/O Wait ⏳ ] ---> [ Task 2 Finish ]
                                                           │
                                                           ▼
[ Task 3 Start ] ---> [ Blocking I/O Wait ⏳ ] ---> [ Task 3 Finish ]

Total Time = Time(Task 1) + Time(Task 2) + Time(Task 3)  (Cumulative)
```

### The Problem: Idle CPU Time
During blocking I/O, your computer's CPU is idle, waiting for network packets or disk drives. If 10 requests take 1 second each, synchronous execution takes **10 seconds total**. Let's demonstrate this in Python code below.


In [ ]:
import time

def fetch_data_sync(task_id: int, delay: int = 1):
    print(f"  🟢 [Sync Task {task_id}] Started (fetching data)...")
    time.sleep(delay)  # Simulates blocking network/disk I/O
    print(f"  ✅ [Sync Task {task_id}] Finished in {delay}s!")
    return f"Data from Task {task_id}"

def run_synchronous_demo():
    print("--- Starting Synchronous Execution ---")
    start_time = time.perf_counter()

    results = []
    for i in range(1, 6):
        data = fetch_data_sync(i, delay=1)
        results.append(data)

    end_time = time.perf_counter()
    total_duration = end_time - start_time
    print(f"\n⏱️ Synchronous Total Execution Time: {total_duration:.2f} seconds")
    print(f"📦 Results collected: {len(results)}")

run_synchronous_demo()


## 2. ⚡ Introduction to `asyncio` & Asynchronous Programming

### What is `asyncio`?
`asyncio` is a built-in Python library introduced in Python 3.4 that provides infrastructure for writing concurrent code using the `async`/`await` syntax.

Instead of waiting passively for an I/O operation to finish, an asynchronous function (**coroutine**) yields control back to a central orchestrator called the **Event Loop**. While one task waits for network I/O, the Event Loop executes other tasks that are ready.

```
Asynchronous Execution Flow (Event Loop):

Event Loop
   │
   ├───> Task 1 Started ---> [ Yields Control on I/O Wait ⏳ ]
   │                                  │ (Event Loop switches to Task 2)
   ├───> Task 2 Started ---> [ Yields Control on I/O Wait ⏳ ]
   │                                  │ (Event Loop switches to Task 3)
   └───> Task 3 Started ---> [ Yields Control on I/O Wait ⏳ ]
                                      │
            (All tasks wait concurrently in parallel-like fashion)
                                      │
   <─── Task 1 I/O Ready ─── Resumes & Finishes
   <─── Task 2 I/O Ready ─── Resumes & Finishes
   <─── Task 3 I/O Ready ─── Resumes & Finishes

Total Time ≈ Max(Time(Task 1), Time(Task 2), Time(Task 3)) + small overhead
```

### Key Components of `asyncio`
1. **Event Loop**: The core loop that manages and distributes execution of tasks, registers I/O callbacks, and pauses/resumes coroutines.
2. **Coroutine (`async def`)**: A function that can pause execution (`await`) and return control to the Event Loop without exiting.
3. **Future**: A low-level object representing an eventual result of an asynchronous operation.
4. **Task**: A wrapped coroutine scheduled to run on the Event Loop concurrently.

### Benefits of `asyncio`
* 🚀 **Massive Performance Boost**: Cuts total wait time for I/O-bound operations from cumulative ($O(N)$) to maximum single wait ($O(1)$).
* 🪶 **Low Memory Footprint**: Unlike multithreading (which requires ~8MB stack allocation per OS thread), tens of thousands of `asyncio` tasks can run in a single OS thread using only a few kilobytes of RAM each.
* 🔒 **No Race Conditions from Preemption**: Threading context switches happen unpredictably at any bytecode instruction. In `asyncio`, context switching ONLY happens explicitly at `await` keywords, making code easier to reason about without heavy lock contention.

---
> 💡 **Google Colab / Jupyter Note**:
> Notebook environments already run an active Event Loop. In standard Python scripts, you enter `asyncio` using `asyncio.run(main())`. In notebooks, top-level `await` is supported directly, or you can use `nest_asyncio` to allow nested `asyncio.run()` calls.


In [ ]:
# Setup nest_asyncio to support asyncio.run() seamlessly in Jupyter / Google Colab
import sys

try:
    import nest_asyncio
    nest_asyncio.apply()
    print("✅ nest_asyncio applied successfully!")
except ImportError:
    print("📦 Installing nest_asyncio for Jupyter environment...")
    !{sys.executable} -m pip install nest_asyncio -q
    import nest_asyncio
    nest_asyncio.apply()
    print("✅ nest_asyncio installed and applied!")


## 3. 🧩 Core Building Blocks: `async`, `await`, and Tasks

Let's look at the basic syntax:
* `async def`: Defines an asynchronous coroutine function instead of a standard function.
* `await`: Suspends execution of the current coroutine until the awaited object finishes, yielding control back to the event loop.
* `asyncio.create_task()`: Schedules a coroutine to run in the background immediately without waiting for it upfront.


In [ ]:
import asyncio

# 1. Defining a Coroutine Function
async def fetch_data_async(task_id: int, delay: int = 1):
    print(f"  🚀 [Async Task {task_id}] Started...")
    await asyncio.sleep(delay)  # NON-BLOCKING asynchronous wait
    print(f"  ✅ [Async Task {task_id}] Finished!")
    return f"Data from Async Task {task_id}"

# 2. Executing Coroutines inside an Async Entrypoint
async def main_basics():
    print("--- Basic Coroutine Execution ---")
    # Sequential await inside async function
    res1 = await fetch_data_async(1, 1)
    res2 = await fetch_data_async(2, 1)
    print(f"Results: {res1}, {res2}\n")

    print("--- Background Task Creation (asyncio.create_task) ---")
    # Schedule task to start running immediately in background
    task3 = asyncio.create_task(fetch_data_async(3, 1))
    task4 = asyncio.create_task(fetch_data_async(4, 1))

    print("  --> Tasks 3 & 4 scheduled in background! Doing other work...")
    await asyncio.sleep(0.2)
    print("  --> Still doing work while tasks run in background...")

    # Await results when needed
    res3 = await task3
    res4 = await task4
    print(f"Background Task Results: {res3}, {res4}")

# Run the entry point
asyncio.run(main_basics())


## 4. 🧰 7 Different Ways to Use `asyncio` in Python

`asyncio` offers versatile patterns for handling various concurrency challenges. Let's explore the 7 primary patterns:

---
### Pattern 1: Concurrent Execution with `asyncio.gather()`
**Use Case**: Run a fixed set of asynchronous tasks simultaneously and wait for all of them to complete, returning results in exact order.


In [ ]:
async def pattern_1_gather():
    print("=== Pattern 1: Concurrent Execution with asyncio.gather() ===")
    start_time = time.perf_counter()

    # Create 5 concurrent coroutines with 1-second delays each
    coroutines = [fetch_data_async(i, delay=1) for i in range(1, 6)]

    # Run all 5 concurrently
    results = await asyncio.gather(*coroutines)

    total_time = time.perf_counter() - start_time
    print(f"\n⏱️ Finished 5 tasks concurrently in: {total_time:.2f} seconds!")
    print(f"📦 Results: {results}\n")

asyncio.run(pattern_1_gather())


---
### Pattern 2: Task Completion Stream with `asyncio.as_completed()` / Structured Concurrency
**Use Case**: Process task results immediately as soon as each individual task finishes, rather than waiting for the slowest task.


In [ ]:
import random

async def fetch_variable_delay(task_id: int):
    delay = random.uniform(0.5, 2.5)
    await asyncio.sleep(delay)
    return task_id, delay

async def pattern_2_as_completed():
    print("=== Pattern 2: Processing Stream with asyncio.as_completed() ===")
    tasks = [fetch_variable_delay(i) for i in range(1, 6)]

    print("Tasks started with random delays. Results processed as they arrive:")
    for completed_task in asyncio.as_completed(tasks):
        task_id, delay = await completed_task
        print(f"  ⚡ Task {task_id} completed first/next after {delay:.2f}s!")
    print()

asyncio.run(pattern_2_as_completed())


---
### Pattern 3: Handling Timeouts & Task Cancellation
**Use Case**: Prevent tasks from hanging indefinitely by enforcing hard timeouts with `asyncio.wait_for()`, and cleanly handle task cancellation.


In [ ]:
async def slow_network_request():
    print("  🌐 Request started...")
    try:
        await asyncio.sleep(5)  # Simulates slow 5-second response
        return "Response Data"
    except asyncio.CancelledError:
        print("  ⚠️ Task was cancelled cleanly during cleanup!")
        raise

async def pattern_3_timeouts():
    print("=== Pattern 3: Timeouts and Cancellation ===")

    # Enforce a 1.5 second timeout on a 5 second request
    try:
        print("Waiting for slow request with 1.5s timeout...")
        result = await asyncio.wait_for(slow_network_request(), timeout=1.5)
    except asyncio.TimeoutError:
        print("  ❌ TimeoutError: Request took too long (>1.5s) and was automatically cancelled!\n")

asyncio.run(pattern_3_timeouts())


---
### Pattern 4: Asynchronous Iterators & Generators (`async for`, `async yield`)
**Use Case**: Stream real-time data, paginated API responses, or large datasets asynchronously without loading everything into memory at once.


In [ ]:
# Asynchronous Generator Function
async def async_data_stream(total_items: int = 4):
    for item in range(1, total_items + 1):
        await asyncio.sleep(0.4)  # Simulate network fetch delay for stream item
        yield f"Streamed Packet #{item}"

async def pattern_4_async_generators():
    print("=== Pattern 4: Async Iterators & Generators ===")
    print("Receiving data stream:")

    # Consuming async stream using 'async for'
    async for packet in async_data_stream(4):
        print(f"  📥 Received: {packet}")
    print()

asyncio.run(pattern_4_async_generators())


---
### Pattern 5: Asynchronous Context Managers (`async with`)
**Use Case**: Safely acquire and release asynchronous resources like HTTP client sessions (`aiohttp`), database connections (`asyncpg`), or locks.


In [ ]:
class AsyncDatabaseConnection:
    def __init__(self, db_name: str):
        self.db_name = db_name

    async def __aenter__(self):
        print(f"  🔌 Connecting to DB '{self.db_name}' asynchronously...")
        await asyncio.sleep(0.3)
        print("  ✅ Connected!")
        return self

    async def execute_query(self, query: str):
        await asyncio.sleep(0.2)
        return f"Results for '{query}'"

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print(f"  🔌 Closing DB connection to '{self.db_name}'...")
        await asyncio.sleep(0.2)
        print("  🔒 Closed safely!")

async def pattern_5_context_managers():
    print("=== Pattern 5: Async Context Managers (async with) ===")
    async with AsyncDatabaseConnection("analytics_db") as db:
        res = await db.execute_query("SELECT COUNT(*) FROM users")
        print(f"  📊 Query Result: {res}")
    print()

asyncio.run(pattern_5_context_managers())


---
### Pattern 6: Producer-Consumer Worker Queue (`asyncio.Queue`)
**Use Case**: Manage rate-limiting, pool background workers, or decouple task generation from task execution.


In [ ]:
async def worker(worker_id: int, queue: asyncio.Queue):
    while True:
        job = await queue.get()
        if job is None:
            # Sentinel signal to shut down worker
            queue.task_done()
            break

        print(f"  👷 Worker {worker_id} processing job '{job}'...")
        await asyncio.sleep(0.5)  # Simulate job processing time
        print(f"  Done processing '{job}'")
        queue.task_done()

async def pattern_6_producer_consumer():
    print("=== Pattern 6: Producer-Consumer Queue ===")
    queue = asyncio.Queue()

    # Enqueue 6 jobs
    for j in range(1, 7):
        await queue.put(f"Job #{j}")

    # Spawn 2 worker tasks
    workers = [asyncio.create_task(worker(w_id, queue)) for w_id in range(1, 3)]

    # Wait until queue is fully processed
    await queue.join()

    # Stop workers by sending sentinel None
    for _ in workers:
        await queue.put(None)
    await asyncio.gather(*workers)
    print("✅ All queued work finished cleanly!\n")

asyncio.run(pattern_6_producer_consumer())


---
### Pattern 7: Running Synchronous/Blocking Code in `asyncio` (`asyncio.to_thread`)
**Use Case**: Execute legacy blocking code (e.g. `requests`, CPU computation, file I/O) without freezing the `asyncio` event loop.


In [ ]:
import time

# A legacy blocking synchronous function
def blocking_cpu_task(name: str, duration: int):
    print(f"  ⚠️ [Sync Task {name}] Blocking thread for {duration}s...")
    time.sleep(duration)
    return f"Result {name}"

async def pattern_7_to_thread():
    print("=== Pattern 7: Running Blocking Code with asyncio.to_thread() ===")
    start = time.perf_counter()

    # asyncio.to_thread runs blocking tasks off the main thread in a separate worker thread
    task_a = asyncio.to_thread(blocking_cpu_task, "A", 2)
    task_b = asyncio.to_thread(blocking_cpu_task, "B", 2)

    # While those run in worker threads, the event loop remains responsive!
    res_a, res_b = await asyncio.gather(task_a, task_b)

    duration = time.perf_counter() - start
    print(f"  ✅ Both blocking tasks completed in parallel threadpool: {duration:.2f}s!")
    print(f"  Outputs: {res_a}, {res_b}\n")

asyncio.run(pattern_7_to_thread())


## 5. 📊 Direct Benchmark: Synchronous vs. Asynchronous Performance

Let's run a side-by-side comparison simulating 10 network API calls (1 second latency each):


In [ ]:
import time
import asyncio

NUM_REQUESTS = 10
DELAY = 1.0

# 1. Sync Benchmark
def sync_benchmark():
    start = time.perf_counter()
    for i in range(NUM_REQUESTS):
        time.sleep(DELAY)
    return time.perf_counter() - start

# 2. Async Benchmark
async def async_fetch():
    await asyncio.sleep(DELAY)

async def async_benchmark():
    start = time.perf_counter()
    tasks = [async_fetch() for _ in range(NUM_REQUESTS)]
    await asyncio.gather(*tasks)
    return time.perf_counter() - start

# Execute Comparison
print("⏳ Running Benchmark (10 requests, 1.0s latency each)...")

sync_duration = sync_benchmark()
print(f"  1️⃣ Synchronous Duration  : {sync_duration:.2f} seconds")

async_duration = asyncio.run(async_benchmark())
print(f"  2️⃣ Asynchronous Duration : {async_duration:.2f} seconds")

speedup = sync_duration / async_duration
print(f"\n🏆 RESULT: Asyncio is {speedup:.1f}x FASTER than Synchronous execution!")


## 6. 🎓 Best Practices, Cheat Sheet & Common Pitfalls

### ❌ Common Pitfalls to Avoid
1. **Calling Synchronous Blocking Functions inside Coroutines**:
   * *Wrong*: Using `time.sleep(1)` or `requests.get()` inside `async def`. This freezes the entire Event Loop and stops all other tasks!
   * *Fix*: Use `await asyncio.sleep(1)`, async libraries (`aiohttp`, `httpx`), or wrap blocking calls with `asyncio.to_thread()`.

2. **Forgetting to `await` Coroutines**:
   * *Wrong*: Calling `fetch_data()` without `await`. It returns an un-awaited coroutine object instead of executing it.
   * *Fix*: Always use `await fetch_data()` or `asyncio.create_task()`.

3. **Using `asyncio` for Heavy CPU-Bound Work**:
   * `asyncio` is designed for **I/O-bound tasks** (network, disk, DB). For heavy mathematical computations or image processing, use Python's `multiprocessing` or `concurrent.futures.ProcessPoolExecutor`.

---

### 📝 Asyncio Usage Cheatsheet

| Task / Goal | Syntax / Method | Description |
| :--- | :--- | :--- |
| **Define Coroutine** | `async def my_func():` | Declares an asynchronous function |
| **Pause & Yield Control** | `await coroutine()` | Suspends execution until result is ready |
| **Start Event Loop** | `asyncio.run(main())` | Entry point for standard scripts |
| **Concurrent Gather** | `await asyncio.gather(t1, t2)` | Runs multiple coroutines concurrently |
| **Background Task** | `task = asyncio.create_task(coro)` | Schedules coroutine to run immediately in background |
| **Enforce Timeout** | `await asyncio.wait_for(coro, 2.0)` | Raises `TimeoutError` if task exceeds timeout |
| **Offload Sync Function**| `await asyncio.to_thread(sync_func)` | Runs blocking sync function in thread pool |
| **Async Context** | `async with Resource() as r:` | Async setup & teardown |
| **Async Generator** | `async for item in stream():` | Stream asynchronous data packets |
| **Task Queue** | `q = asyncio.Queue()` | Producer-consumer concurrency queue |

---
## 🎉 Congratulations!
You have completed the **Asyncio & Concurrency Mastery** notebook. You now possess a deep understanding of Python's event loop, synchronous vs. asynchronous execution, and how to harness `asyncio` to write lightning-fast, concurrent applications!
